# OpenPowerlifting — Statistical-Theory Results

**How lifters "game" the two numbers they control — the weight on the bar and their bodyweight — and what ~3.94M competition results reveal about the limits of strength.**

Authors: Adir Elmakais, David Levin · Bar-Ilan MSc, Statistical Theory.

This notebook presents the results. The analysis itself lives in `src/` (run `python src/h1_quantization.py`, `h2_bunching.py`, `h3_allometry.py`, `h4_prediction.py`, `h5_supporting.py`, `breadth_tools.py`, then `figures.py`); numbers are saved to `results/*.json` and figures to `figures/`. Every result is computed on the locked data snapshot (3,941,811 rows, CC0).


In [ ]:
import json, pathlib
from IPython.display import Image, display
ROOT = pathlib.Path.cwd(); ROOT = ROOT if (ROOT/'results').exists() else ROOT.parent
R = lambda n: json.loads((ROOT/'results'/n).read_text())
FIG = lambda n: display(Image(filename=str(ROOT/'figures'/n)))
h1,h2,h3,h4,h5,br = R('h1.json'),R('h2.json'),R('h3.json'),R('h4.json'),R('h5_supporting.json'),R('breadth.json')
print('loaded results for H1-H5 + breadth')


## H1 — Attempt loads are quantized to the 2.5 kg plate grid
Lead with the effect size; the formal test is a GLRT whose null parameter sits on the boundary, so its null distribution is calibrated by a parametric bootstrap (Wilks' chi-squared is invalid there).


In [ ]:
print(f"on-grid: {h1['effect_size_on_grid_pct']}%  CI {h1['on_grid_ci_pct']}  (n={h1['n_valid_attempts']:,})")
print(f"GLRT 2lnLambda = {h1['glrt_2lnLambda']:,.0f}; bootstrap null mass@0 = {h1['bootstrap_null_mass_at_0']} "
      f"(50:50 boundary mixture); bootstrap p < {h1['bootstrap_p_upper']:.1e}")
print(f"off-grid loads are mostly pounds: {h1['lb_robustness']['round_lb_share_of_off']*100:.0f}% of the "
      f"{h1['lb_robustness']['off_lattice_frac']*100:.1f}% off-grid are round-lb loads")
FIG('fig1_quantization.png')


## H2 — Bodyweight bunches just below class limits (weight-cutting)
Effect sizes are de-heaped log(below/above) on all rows; the formal Cattaneo-Jansson-Ma density test (rddensity) runs on one random meet per lifter, pure-kg post-2014. Falsification: a non-limit control and placebos.


In [ ]:
e83=h2['effect_sizes']['83']; print(f"83 kg: log-ratio {e83['log_ratio']} +/- {e83['ci_half']} (x{e83['ratio']}); "
      f"control 91: {h2['effect_sizes']['91']['log_ratio']}")
print('all 7 limits reject after Holm?', h2['multiple_testing']['all_limits_reject_holm'],
      '| BH?', h2['multiple_testing']['all_limits_reject_bh'])
print('placebo false-positives (bunching direction):', h2['multiple_testing']['placebo_false_positives_bunching_dir'] or 'NONE')
FIG('fig2_bunching.png'); FIG('fig3_all_limits.png')


## H3 — Allometric scaling differs by sex
log-log OLS with HC3 robust SE on one meet per lifter; Wald test of b vs 2/3; a Sex x log(BW) interaction tests the difference formally. The sex gap persists on common bodyweight support (not a range artifact). Fig 4 plots the all-row descriptive OLS fit (0.72 / 0.49); the formal per-lifter HC3 estimates are 0.75 / 0.51.


In [ ]:
for s in ('men','women'):
    f=h3['by_sex'][s]['formal_per_lifter']; w=h3['by_sex'][s]['wald_vs_2/3']
    print(f"{s}: b={f['b']} CI{f['ci']}  Wald vs 2/3 z={w['z']}")
print('sex interaction (men-women slope):', h3['sex_interaction']['coef_male_x_logbw'])
print('diagnosis:', h3['women_b_diagnosis']['note'])
FIG('fig4_allometry.png')


## H4 — Predicting strength (the learning half)
Leakage-safe: every split grouped by lifter, fold-local preprocessing. Random Forest vs linear, R2/RMSE; a logistic 'made-weight' classifier on non-bodyweight features only (Dots excluded as circular).


In [ ]:
for m,v in h4['regression_total']['models'].items():
    print(f"{m}: CV R2={v['cv_r2']}  RMSE={v.get('cv_rmse_kg')} kg")
c=h4['cut_classifier']; print(f"cut classifier AUC={c['auc']} PR-AUC={c['pr_auc']} (Dots excluded={c['dots_excluded']})")
FIG('fig5_prediction.png')


## Supporting analyses
Distribution structure (the apparent mixture is just sex), tested-vs-untested, EVT upper-tail (a ceiling is NOT supported -- the xi CI includes 0), time-trend.


In [ ]:
ds=h5['distribution_structure']; print('best #components by BIC -- Dots:', ds['dots_normalized']['best_k_by_bic'],
      '| raw Total:', ds['raw_total']['best_k_by_bic'])
t=h5['tested_vs_untested']; print(f"tested vs untested Dots: median {t['median_tested']} vs {t['median_untested']}, rank-biserial {t['rank_biserial']}")
ev=h5['evt_gev']; print(f"EVT: xi={ev['xi']} CI{ev['xi_ci']} -> {ev['tail']}")
tr=h5['time_trend']; print(f"time-trend median Dots: Spearman rho={tr['spearman_rho']} ({tr['direction']})")
FIG('fig6_structure.png')


## Breadth — full course toolbox
Paired Wilcoxon, ANOVA/Kruskal-Wallis + post-hoc, independence chi-square + Cramer's V, a sequential Wald test (SPRT) with a stopping time, an a-priori power analysis (Type I/II tradeoff), all four multiple-testing corrections, and the MP/UMP framing.


In [ ]:
print('paired Wilcoxon (Squat opener vs final): diff', br['paired_attempts']['median_diff_kg'],'kg, p',br['paired_attempts']['wilcoxon_p'])
print('ANOVA equipment F=',br['anova_equipment']['anova_F'],'; Kruskal H=',br['anova_equipment']['kruskal_H'])
ic=br['independence_cut_tested']; print(f"independence chi2={ic['chi2']} p={ic['p']:.3f} Cramers V={ic['cramers_v']} (negligible)")
sp_=br['sprt_cut']; print(f"SPRT decision '{sp_['decision']}' at stopping time N={sp_['stopping_time']}")
print('power: n for 0.90 =', br['power_analysis']['n_for_power_0.90'])
print('all-4 corrections on 7 limits:', {m:d['n_rejected'] for m,d in br['all_four_corrections']['corrections'].items()})
FIG('fig7_breadth.png')


## Reproducibility
Clone the repo, `pip install -r requirements.txt`, download the pinned data snapshot (`python download_data.py`; SHA256 recorded in `src/config.py`), then run the `src/` scripts and `src/figures.py`. The numbers above are read from `results/*.json`; an independent recomputation from the raw CSV matched every headline value.
